# Phase 6 — Autoencoder Explainability Training
**RealCentric Generator-Agnostic Deepfake Detection**  
SVKM AI/ML HPC Cluster

Trains a convolutional autoencoder on **real images only**.  
Generates pixel-level heatmaps: `R(x) = |x − g(x)|`  

**Input:** `/data/mpstme-naman/deepfake_detection/data/processed/celebdf/real/`  
**Output:** `/data/mpstme-naman/deepfake_detection/checkpoints/autoencoder_best.pt`  
**Estimated time:** 30–60 minutes

## Step 1 — Load Real Images

In [1]:
import sys, cv2, numpy as np, torch
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path
from tqdm import tqdm

BASE     = Path('/data/mpstme-naman/deepfake_detection')
PROC     = BASE / 'data' / 'processed'
CKPT_DIR = BASE / 'checkpoints'
RES_DIR  = BASE / 'results'

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
paths = sorted((PROC/'celebdf'/'real').glob('*.png'))[:20000]
print(f'Loading {len(paths):,} real images...')
real_images = [cv2.imread(str(p)) for p in tqdm(paths, desc='Loading')]
real_images = [x for x in real_images if x is not None]
print(f'  Loaded: {len(real_images):,}  shape={real_images[0].shape}')

GPU: NVIDIA H100 PCIe MIG 3g.40gb
Loading 20,000 real images...


Loading: 100%|██████████| 20000/20000 [00:41<00:00, 485.99it/s]

  Loaded: 20,000  shape=(256, 256, 3)


## Step 2 — Initialise and Train

In [2]:
from config.config_loader import load_config
from src.models.autoencoder import DeepfakeExplainer

cfg = load_config()
explainer = DeepfakeExplainer(cfg=cfg)

print('Architecture  : 256×256×3 → bottleneck 32×32×64 → 256×256×3')
print('Trained on    : REAL images only  (30 epochs)')
print()

explainer.train_autoencoder(real_images, checkpoint_dir=str(CKPT_DIR), run_name='autoencoder')
print(f'  ✓  Saved → {CKPT_DIR}/autoencoder_best.pt')

Architecture  : 256×256×3 → bottleneck 32×32×64 → 256×256×3
Trained on    : REAL images only  (30 epochs)


  Autoencoder Training  [256×256]
  Device: cuda   Epochs: 30   Batch: 16
  Real images: 20000
   Epoch        Loss  Status
  --------------------------------
       1    0.003702  ← best
       2    0.001419  ← best
       3    0.001049  ← best
       4    0.000898  ← best
       5    0.000792  ← best
       6    0.000692  ← best
       7    0.000646  ← best
       8    0.000609  ← best
       9    0.000559  ← best
      10    0.000539  ← best
      11    0.000490  ← best
      12    0.000480  ← best
      13    0.000437  ← best
      14    0.000433  ← best
      15    0.000411  ← best
      16    0.000397  ← best
      17    0.000368  ← best
      18    0.000341  ← best
      19    0.000343  
      20    0.000329  ← best
      21    0.000318  ← best
      22    0.000304  ← best
      23    0.000303  ← best
      24    0.000286  ← best
      25    0.000278  ← best
      26    0.

## Step 3 — Test on Real Image (expect LOW heatmap)

In [ ]:
import matplotlib.pyplot as plt
res = explainer.explain(real_images[42])
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Real Image — Heatmap Should Be LOW', fontsize=12)
axes[0].imshow(cv2.cvtColor(real_images[42], cv2.COLOR_BGR2RGB)); axes[0].set_title('Original'); axes[0].axis('off')
im = axes[1].imshow(res['heatmap'], cmap='jet', vmin=0, vmax=0.3)
axes[1].set_title(f'Heatmap  score={res["score"]:.4f}'); axes[1].axis('off'); plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].imshow(cv2.cvtColor(res['overlay'], cv2.COLOR_BGR2RGB)); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.tight_layout(); plt.savefig(RES_DIR/'ae_real_example.png', dpi=120, bbox_inches='tight'); plt.show()
print(f'  Real score: {res["score"]:.4f}')

## Step 4 — Test on Fake Image (expect HIGH heatmap)

In [ ]:
fp  = sorted((PROC/'celebdf'/'fake').glob('*.png'))[42]
img = cv2.imread(str(fp))
res = explainer.explain(img)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Fake Image — Heatmap Should Be HIGH', fontsize=12, color='#d32f2f')
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original Fake'); axes[0].axis('off')
im = axes[1].imshow(res['heatmap'], cmap='jet', vmin=0, vmax=0.3)
axes[1].set_title(f'Heatmap  score={res["score"]:.4f}'); axes[1].axis('off'); plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].imshow(cv2.cvtColor(res['overlay'], cv2.COLOR_BGR2RGB)); axes[2].set_title('Bright = Suspicious Region'); axes[2].axis('off')
plt.tight_layout(); plt.savefig(RES_DIR/'ae_fake_example.png', dpi=120, bbox_inches='tight'); plt.show()
print(f'  Fake score: {res["score"]:.4f}  (should be higher than real)')

## ✅ Phase 6 Complete

**Saved:** `checkpoints/autoencoder_best.pt`

**Next:** Open `06_benchmark.ipynb`